# Security Selection

In [1]:
import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
from src.selection import (
    FundamentalSelector, 
    FundamentalScoreConfig, 
    MetricSignalSpec
)

# 1
from src.selection import (
    FundamentalLearningPanelBuilder,
    YahooFundamentalsProvider,
)

# 2 
from src.selection import XGBoostFundamentalModel

# 3
from src.selection import LearnedScoreConfigFactory

# 4. Rankear Empresas Con El Score Aprendido
from src.selection import LearnedFundamentalSelector


## Scoring

In [28]:
document_path = "../data/raw/screening_documents/Companyscreenings_Results_report_food_staples_retail.xlsx"

In [37]:
df_tickets = pd.read_excel(io = document_path, header=3)

df_tickets['Identifier'] = df_tickets['Identifier'].apply(lambda ticket : ticket.split(sep= "-")[0])

df_tickets.head(3)

,Identifier,Name,Company Type Main,Company Country,Year Founded,Number of Employees,Revenue (MM) (USD),Company Region,FactSet RBICS Rev Sector,Price,...,Return on Avg Invest Capital (USD),Gross Profit Growth (USD),Gross Profit CAGR 3-year (USD),Gross Income Margin (USD),Net Income Margin (USD),Net Income CAGR 3-year (USD),EBITDA Margin (USD),Total Debt/EBITDA (USD),EPS CAGR 3-year (USD),Earnings Per Share (EPS) (Diluted) (USD)
0,CRDE,Cardinal Ethanol LLC,Public,United States,-,72,495.861681,North America,"Chemical, Plastic and Rubber Materials|Food an...",22750,...,38.587638,-17.71427,119.901,16.915692,14.080255,-,17.422667,0.353839,-,4780.1308
1,DDS,"Dillard's, Inc.",Public,United States,1938,29100,6563.336000,North America,Food and Staples Retail|Industrial Services,605.77,...,24.947709,-1.030241,-4.40536,37.589619,8.687457,-13.8458,13.22521,0.600997,-10.5028,36.422
2,BRK.B,"Berkshire Hathaway, Inc.",Public,United States,1839,387815,371444.000000,North America,Food and Staples Retail|Industrial Services|Ma...,474.58,...,8.33183,1.382568,13.905,23.630749,18.029097,-,19.479652,1.786279,-,31.042


In [38]:
df_tickets.info()

<class 'pandas.DataFrame'>
RangeIndex: 163 entries, 0 to 162
Data columns (total 27 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   Identifier                                163 non-null    str    
 1   Name                                      163 non-null    str    
 2   Company Type Main                         163 non-null    str    
 3   Company Country                           163 non-null    str    
 4   Year Founded                              163 non-null    object 
 5   Number of Employees                       163 non-null    object 
 6   Revenue (MM) (USD)                        163 non-null    float64
 7   Company Region                            163 non-null    str    
 8   FactSet RBICS Rev Sector                  163 non-null    str    
 9   Price                                     163 non-null    object 
 10  Price to Earnings (P/E)                   163 non

In [41]:
df_tickets_filter1 = df_tickets['Company Region'] == 'North America'

palabras = 'Tobacco'

# df_tickets_filter2 = df_tickets['FactSet RBICS Rev Sector'].str.contains(
#     palabras, case = False, na = False
# ) 

df_tickets_filter3 = df_tickets['Year Founded'].apply(lambda year : int(year) if year != "-" else 0) <= 2024

df_tickets_us = df_tickets.loc[df_tickets_filter1 & df_tickets_filter3, :]

In [42]:
ticker_us = df_tickets_us.loc[:, "Identifier"].values.tolist()

len(ticker_us)

163

In [8]:
provider = YahooFundamentalsProvider(
    start="2018-01-01",
    end="2026-05-01",
)

panel_builder = FundamentalLearningPanelBuilder(
    provider=provider,
    frequency="quarterly",
    trailing_periods=20,
    horizon_months=12,
    reporting_lag_days=60,
)

panel = panel_builder.build(ticker_us)

/home/rigodev/ITESO/06/portafolios/env/lib/python3.12/site-packages/yfinance/scrapers/quote.py:702: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  start = pd.Timestamp.utcnow().floor("D") - datetime.timedelta(days=365 // 2)
/home/rigodev/ITESO/06/portafolios/env/lib/python3.12/site-packages/yfinance/scrapers/quote.py:704: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  end = pd.Timestamp.utcnow().ceil("D")
/home/rigodev/ITESO/06/portafolios/env/lib/python3.12/site-packages/yfinance/scrapers/history.py:201: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
/home/rigodev/ITESO/06/portafolios/env/lib/python3.12/site-packages/yfinance/scrapers/fundamentals.py:120: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a f

In [11]:
panel

,ticker,frequency,period,current_price,market_cap,enterprise_value,shares_outstanding,eps,eps_recurring,eps_basic,...,free_cash_flow__yoy_change,free_cash_flow__historical_avg_change,available_at,target_end_at,entry_price,exit_price,forward_dividends_12m,forward_price_return_12m,forward_dividend_return_12m,forward_total_return_12m
0,ABT,quarterly,2024-09-30,90.790001,1.581392e+11,NaN,1.741813e+09,NaN,NaN,NaN,...,NaN,NaN,2024-11-29,2025-11-29,115.250839,126.570763,2.36,0.098220,0.020477,0.118697
1,ABT,quarterly,2024-12-31,90.790001,1.581392e+11,NaN,1.741813e+09,NaN,NaN,NaN,...,NaN,NaN,2025-03-01,2026-03-01,136.779297,113.408516,2.40,-0.170865,0.017547,-0.153318
2,ABT,quarterly,2025-03-31,90.790001,1.579598e+11,1.643578e+11,1.739837e+09,0.76,0.76,0.760000,...,NaN,NaN,2025-05-30,2026-05-30,130.905823,NaN,2.44,NaN,0.018639,NaN
3,ABT,quarterly,2025-06-30,90.790001,1.580163e+11,1.641713e+11,1.740459e+09,1.01,1.01,1.022144,...,NaN,0.655949,2025-08-29,2026-08-29,130.587799,NaN,1.85,NaN,0.014167,NaN
4,ABT,quarterly,2025-09-30,90.790001,1.578722e+11,1.630802e+11,1.738872e+09,0.94,0.94,0.945441,...,NaN,0.569398,2025-11-29,2026-11-29,126.570763,NaN,1.26,NaN,0.009955,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
619,ZVIA,quarterly,2024-12-31,1.280000,7.890749e+07,4.956749e+07,6.164648e+07,-0.09,-0.09,-0.090000,...,NaN,NaN,2025-03-01,2026-03-01,2.330000,1.390000,0.00,-0.403433,0.000000,-0.403433
620,ZVIA,quarterly,2025-03-31,1.280000,8.456275e+07,5.809175e+07,6.606465e+07,-0.08,-0.08,-0.080000,...,NaN,-0.442752,2025-05-30,2026-05-30,2.720000,NaN,0.00,NaN,0.000000,NaN
621,ZVIA,quarterly,2025-06-30,1.280000,9.593548e+07,7.072048e+07,7.494959e+07,-0.01,-0.01,-0.010000,...,NaN,0.036628,2025-08-29,2026-08-29,2.800000,NaN,0.00,NaN,0.000000,NaN
622,ZVIA,quarterly,2025-09-30,1.280000,8.623579e+07,6.103679e+07,6.737171e+07,-0.04,-0.04,-0.040000,...,NaN,0.342505,2025-11-29,2026-11-29,2.590000,NaN,0.00,NaN,0.000000,NaN


In [12]:
model = XGBoostFundamentalModel(
    target="forward_total_return_12m",
    group_by="sector",
    min_group_size=30,
    min_feature_coverage=0.50,
    n_splits=3,
)


In [13]:
model.fit(panel)

In [14]:
model.training_summary_


,group,n_obs,n_features,oos_score,status
0,Consumer Defensive,167,12,0.251940,fit
1,__global__,209,12,0.300148,fit


In [15]:
impact = model.impact_report_
impact

,group,feature,metric,signal,importance,higher_is_better,stability,oos_score,coverage,adjusted_impact,n_obs,model
0,Consumer Defensive,trailing_pe__level,trailing_pe,level,0.045931,True,0.826700,0.251940,0.824950,0.007892,167,XGBRegressor
1,Consumer Defensive,eps_recurring__level,eps_recurring,level,0.043992,True,0.867833,0.251940,0.828974,0.007974,167,XGBRegressor
2,Consumer Defensive,diluted_shares_outstanding__level,diluted_shares_outstanding,level,0.078774,True,0.718851,0.251940,1.000000,0.014267,167,XGBRegressor
3,Consumer Defensive,diluted_shares_outstanding__period_change,diluted_shares_outstanding,period_change,0.047078,True,0.366025,0.251940,0.839034,0.003643,167,XGBRegressor
4,Consumer Defensive,diluted_shares_outstanding__historical_avg_change,diluted_shares_outstanding,historical_avg_change,0.078811,True,0.366025,0.251940,0.839034,0.006098,167,XGBRegressor
5,Consumer Defensive,basic_shares_outstanding__level,basic_shares_outstanding,level,0.068147,True,0.848552,0.251940,1.000000,0.014569,167,XGBRegressor
6,Consumer Defensive,basic_shares_outstanding__period_change,basic_shares_outstanding,period_change,0.076452,True,0.366025,0.251940,0.839034,0.005915,167,XGBRegressor
7,Consumer Defensive,basic_shares_outstanding__historical_avg_change,basic_shares_outstanding,historical_avg_change,0.201893,True,1.000000,0.251940,0.839034,0.042677,167,XGBRegressor
8,Consumer Defensive,total_shares_outstanding__level,total_shares_outstanding,level,0.053255,True,0.756878,0.251940,1.000000,0.010155,167,XGBRegressor
9,Consumer Defensive,total_shares_outstanding__period_change,total_shares_outstanding,period_change,0.092703,True,1.000000,0.251940,0.839034,0.019596,167,XGBRegressor


### Pesos

In [19]:
factory = LearnedScoreConfigFactory(
    max_features=15,
    min_impact=0.0,
    normalizer="percentile_rank",
)

configs = factory.from_impact_report(impact)

### Rank

In [21]:
selector = LearnedFundamentalSelector(
    score_configs_by_group=configs,
    provider=provider,
    group_by="sector",
)

ranking = selector.rank(
    ticker_us,
    frequency="quarterly",
    trailing_periods=8,
)

In [23]:
selected = selector.select_top(ranking, top_k=5)
selected

,ticker,current_price,market_cap,enterprise_value,shares_outstanding,eps,eps_recurring,eps_basic,eps_diluted,revenue,...,basic_shares_outstanding__period_change_score,diluted_shares_outstanding__historical_avg_change,diluted_shares_outstanding__historical_avg_change_score,diluted_shares_outstanding__period_change,diluted_shares_outstanding__period_change_score,fundamental_score,score_coverage,score_group,strategy,rank_in_group
0,VLO,251.7800,7.528883e+10,8.212580e+10,298953671.0,13.70,13.70,309000000.0,7.570000,1.226870e+11,...,1.000000,2.911757e-04,1.000000,0.000000,1.000000,100.000000,1.0,Energy,learned_global,1
1,BLNE,2.0200,6.190768e+07,7.085842e+07,30647369.0,-1.60,-1.60,14144679.0,-2.230000,8.202000e+06,...,1.000000,3.305433e+00,1.000000,0.413081,1.000000,100.000000,1.0,Financial Services,learned_global,1
2,TULP,3.9439,7.439690e+06,9.061162e+07,1886379.0,-1.38,-1.38,1883060.0,-3.045467,3.777300e+07,...,1.000000,7.898394e-07,1.000000,0.001989,1.000000,100.000000,1.0,Communication Services,learned_global,1
3,LAND,9.5900,4.108027e+08,9.303006e+08,42836573.0,-0.29,-0.29,36506720.0,-0.290000,8.833900e+07,...,1.000000,-2.097286e-02,1.000000,0.047029,1.000000,95.555823,1.0,Real Estate,learned_global,1
4,BYND,0.9390,4.352843e+08,7.387969e+08,463561581.0,-1.83,-1.83,155266711.0,-1.830000,2.754960e+08,...,0.981013,8.168367e-01,0.981013,4.911100,0.981013,84.977379,1.0,Consumer Defensive,learned_consumer_defensive,1


In [24]:
selected_by_sector = selector.select_top(
    ranking,
    top_k=2,
    per_group=True,
)

selected_by_sector

,ticker,current_price,market_cap,enterprise_value,shares_outstanding,eps,eps_recurring,eps_basic,eps_diluted,revenue,...,basic_shares_outstanding__period_change_score,diluted_shares_outstanding__historical_avg_change,diluted_shares_outstanding__historical_avg_change_score,diluted_shares_outstanding__period_change,diluted_shares_outstanding__period_change_score,fundamental_score,score_coverage,score_group,strategy,rank_in_group
0,AMTX,3.5800,2.439612e+08,7.437381e+08,6.814559e+07,-1.28,-1.28,5.998200e+07,-1.280000,2.079810e+08,...,1.000000,4.226312e-03,1.000000,0.018292,1.000000,80.956824,1.000000,Basic Materials,learned_global,1
1,SXT,114.5550,4.874466e+09,5.505509e+09,4.255132e+07,3.39,3.39,4.223600e+07,3.160000,1.612111e+09,...,0.857143,1.575800e-05,0.428571,0.006678,0.857143,67.649673,1.000000,Basic Materials,learned_global,2
2,TULP,3.9439,7.439690e+06,9.061162e+07,1.886379e+06,-1.38,-1.38,1.883060e+06,-3.045467,3.777300e+07,...,1.000000,7.898394e-07,1.000000,0.001989,1.000000,100.000000,1.000000,Communication Services,learned_global,1
3,SBUX,104.9730,1.196377e+11,1.434098e+11,1.139700e+09,1.31,1.31,1.136900e+09,1.630000,3.718440e+10,...,0.600000,-2.812446e-05,0.600000,0.000351,0.600000,77.432546,1.000000,Consumer Cyclical,learned_global,1
4,CAKE,59.2900,2.956146e+09,4.915623e+09,4.979600e+07,3.41,3.41,4.678600e+07,3.060000,3.751806e+09,...,1.000000,1.870862e-04,0.800000,0.003226,1.000000,75.651167,1.000000,Consumer Cyclical,learned_global,2
5,BYND,0.9390,4.352843e+08,7.387969e+08,4.635616e+08,-1.83,-1.83,1.552667e+08,-1.830000,2.754960e+08,...,0.981013,8.168367e-01,0.981013,4.911100,0.981013,84.977379,1.000000,Consumer Defensive,learned_consumer_defensive,1
6,BG,127.6200,2.476059e+10,4.019860e+10,1.940181e+08,3.80,3.80,1.934087e+08,4.219046,7.032900e+10,...,0.835443,2.246831e-02,0.860759,0.003136,0.835443,82.821121,1.000000,Consumer Defensive,learned_consumer_defensive,2
7,VLO,251.7800,7.528883e+10,8.212580e+10,2.989537e+08,13.70,13.70,3.090000e+08,7.570000,1.226870e+11,...,1.000000,2.911757e-04,1.000000,0.000000,1.000000,100.000000,1.000000,Energy,learned_global,1
8,BLNE,2.0200,6.190768e+07,7.085842e+07,3.064737e+07,-1.60,-1.60,1.414468e+07,-2.230000,8.202000e+06,...,1.000000,3.305433e+00,1.000000,0.413081,1.000000,100.000000,1.000000,Financial Services,learned_global,1
9,ABT,87.5800,1.525480e+11,1.833035e+11,1.741813e+09,3.57,3.57,1.736599e+09,3.720000,4.432800e+10,...,1.000000,1.079378e-06,0.333333,0.003002,1.000000,73.466208,1.000000,Healthcare,learned_global,1


In [27]:
report = selector.selection_report(
    ranking,
    top_k=5,
)

report["selected"]

,ticker,fundamental_score,score_coverage,score_group,rank_in_group,strategy,sector,industry
0,VLO,100.000000,1.0,Energy,1,learned_global,Energy,Oil & Gas Refining & Marketing
1,BLNE,100.000000,1.0,Financial Services,1,learned_global,Financial Services,Mortgage Finance
2,TULP,100.000000,1.0,Communication Services,1,learned_global,Communication Services,Advertising Agencies
3,LAND,95.555823,1.0,Real Estate,1,learned_global,Real Estate,REIT - Specialty
4,BYND,84.977379,1.0,Consumer Defensive,1,learned_consumer_defensive,Consumer Defensive,Packaged Foods


## Value

In [46]:
selector_us = FundamentalSelector(
    strategy = "value"
)

selector_us.rank(
    tickers=ticker_us,
    frequency='annual'
)

$BF.B: possibly delisted; no price data found  (1d 2020-01-01 -> 2026-05-04)


,ticker,current_price,market_cap,enterprise_value,shares_outstanding,eps,eps_recurring,eps_basic,eps_diluted,revenue,...,revenue__historical_avg_change_score,eps__historical_avg_change,eps__historical_avg_change_score,free_cash_flow_margin__historical_avg_change,free_cash_flow_margin__historical_avg_change_score,debt_to_equity__historical_avg_change,debt_to_equity__historical_avg_change_score,fundamental_score,score_coverage,strategy
0,CALM,76.2500,3.612465e+09,2.467502e+09,47376588.0,14.370,14.370,48719000.0,24.95,4.261885e+09,...,0.975490,2.485797,0.893204,0.073085,0.892157,NaN,0.00,81.647759,0.97,value
1,LOCL,2.0200,4.604630e+07,5.765346e+08,22795198.0,-5.610,-5.610,8480247.0,-14.14,4.836500e+07,...,0.975490,0.162558,0.533981,1.947212,0.975490,-1.299230,0.95,68.191991,0.96,value
2,BYND,0.9487,4.397809e+08,7.387969e+08,463561581.0,-1.830,-1.830,155266711.0,-1.83,2.754960e+08,...,0.078431,0.290051,0.601942,0.122879,0.921569,-167.878609,0.98,66.087402,0.96,value
3,VFF,2.8500,3.257228e+08,2.957296e+08,114288686.0,0.180,0.180,111865517.0,0.27,2.159370e+08,...,0.176471,0.853521,0.766990,0.086834,0.901961,-0.034795,0.69,65.459755,0.96,value
4,LFVN,5.4300,6.952991e+07,6.997794e+07,12804772.0,0.600,0.600,12458000.0,0.23,2.285300e+08,...,0.598039,0.419979,0.669903,0.004941,0.382353,-0.054263,0.76,65.019675,1.00,value
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
102,SDSYA,7.0000,2.129330e+08,7.199210e+08,30419000.0,0.695,0.695,NaN,NaN,5.038151e+08,...,0.098039,-0.264830,0.233010,-0.168593,0.034314,0.578288,0.09,31.322216,0.90,value
103,MSTH,0.0110,1.278555e+06,1.156241e+07,116030948.0,NaN,NaN,NaN,NaN,NaN,...,0.000000,NaN,0.000000,NaN,0.000000,NaN,0.00,14.305515,0.21,value
104,CHSCP,27.6100,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000000,NaN,0.000000,NaN,0.000000,NaN,0.00,5.747600,0.09,value
105,BWEL,557.5000,5.375471e+08,NaN,964210.0,NaN,NaN,NaN,NaN,NaN,...,0.000000,NaN,0.000000,NaN,0.000000,NaN,0.00,3.544522,0.09,value


In [47]:
report_value = selector_us.selection_report()
report_value["selected"]

,ticker,fundamental_score,score_coverage,strategy,sector,industry
0,CALM,81.647759,0.97,value,Consumer Defensive,Farm Products
1,LOCL,68.191991,0.96,value,Consumer Defensive,Farm Products
2,BYND,66.087402,0.96,value,Consumer Defensive,Packaged Foods
3,VFF,65.459755,0.96,value,Consumer Defensive,Farm Products
4,LFVN,65.019675,1.00,value,Consumer Defensive,Packaged Foods


In [48]:
selector_us.select_top(top_k = 60)

,ticker,current_price,market_cap,enterprise_value,shares_outstanding,eps,eps_recurring,eps_basic,eps_diluted,revenue,...,revenue__historical_avg_change_score,eps__historical_avg_change,eps__historical_avg_change_score,free_cash_flow_margin__historical_avg_change,free_cash_flow_margin__historical_avg_change_score,debt_to_equity__historical_avg_change,debt_to_equity__historical_avg_change_score,fundamental_score,score_coverage,strategy
0,CALM,76.2500,3.612465e+09,2.467502e+09,4.737659e+07,14.37,14.37,4.871900e+07,24.950000,4.261885e+09,...,0.975490,2.485797,0.893204,0.073085,0.892157,NaN,0.00,81.647759,0.97,value
1,LOCL,2.0200,4.604630e+07,5.765346e+08,2.279520e+07,-5.61,-5.61,8.480247e+06,-14.140000,4.836500e+07,...,0.975490,0.162558,0.533981,1.947212,0.975490,-1.299230,0.95,68.191991,0.96,value
2,BYND,0.9487,4.397809e+08,7.387969e+08,4.635616e+08,-1.83,-1.83,1.552667e+08,-1.830000,2.754960e+08,...,0.078431,0.290051,0.601942,0.122879,0.921569,-167.878609,0.98,66.087402,0.96,value
3,VFF,2.8500,3.257228e+08,2.957296e+08,1.142887e+08,0.18,0.18,1.118655e+08,0.270000,2.159370e+08,...,0.176471,0.853521,0.766990,0.086834,0.901961,-0.034795,0.69,65.459755,0.96,value
4,LFVN,5.4300,6.952991e+07,6.997794e+07,1.280477e+07,0.60,0.60,1.245800e+07,0.230000,2.285300e+08,...,0.598039,0.419979,0.669903,0.004941,0.382353,-0.054263,0.76,65.019675,1.00,value
5,CAG,14.0600,6.726826e+09,1.400413e+10,4.784372e+08,-0.10,-0.10,4.783000e+08,2.400000,1.161280e+10,...,0.411765,0.537372,0.718447,0.016790,0.598039,-0.047132,0.72,63.553562,1.00,value
6,AAGR,0.0099,5.728810e+05,1.220510e+07,5.786683e+07,-0.80,-0.80,3.572063e+07,-1.210000,1.817375e+06,...,0.975490,-4.184791,0.067961,2.322797,0.975490,0.382884,0.13,62.643957,0.96,value
7,CLX,87.1100,1.053263e+10,1.399146e+10,1.209119e+08,6.15,6.15,1.241740e+08,2.250000,7.104000e+09,...,0.382353,0.449824,0.679612,0.010615,0.470588,1.130342,0.06,62.396213,1.00,value
8,BRBR,17.2000,2.017036e+09,3.171536e+09,1.172695e+08,1.45,1.45,1.269000e+08,1.680000,2.316600e+09,...,0.901961,0.271049,0.592233,0.032155,0.803922,0.027303,0.38,62.270144,0.96,value
9,VITL,14.1800,6.075911e+08,5.753633e+08,4.284845e+07,1.44,1.44,4.284966e+07,1.180000,7.594440e+08,...,0.941176,4.855142,0.932039,-0.003996,0.274510,0.028249,0.37,60.928748,0.96,value


## Grow

In [49]:
selector_us.set_strategy(strategy = 'growth')

rank_growth = selector_us.rank(tickers = ticker_us)

rank_growth_over_time = selector_us.rank_over_time(tickers = ticker_us)

rank_growth_evolution = selector_us.metric_evolution(tickers = ticker_us)

In [43]:
report_growth = selector_us.selection_report(ranking = rank_growth, top_k = 10)

In [42]:
report_growth['selected']

,ticker,fundamental_score,score_coverage,strategy,sector,industry
0,COCO,82.694809,0.98,growth,Consumer Defensive,Beverages - Non-Alcoholic
1,LAND,76.667987,0.95,growth,Real Estate,REIT - Specialty
2,MKC,72.499620,1.00,growth,Consumer Defensive,Packaged Foods
3,CELH,72.136250,1.00,growth,Consumer Defensive,Beverages - Non-Alcoholic
4,MAMA,71.951203,1.00,growth,Consumer Defensive,Packaged Foods
5,PAHC,70.933916,1.00,growth,Healthcare,Drug Manufacturers - Specialty & Generic
6,HSY,67.883021,1.00,growth,Consumer Defensive,Confectioners
7,MZTI,67.032444,0.98,growth,Consumer Defensive,Packaged Foods
8,STKL,66.977836,1.00,growth,Consumer Defensive,Beverages - Non-Alcoholic
9,VITL,66.223091,1.00,growth,Consumer Defensive,Farm Products


In [ ]:
rank_growth_evolution.fillna(0).round(decimals = 4)

period,2024-09-30,2024-12-31,2025-01-31,2025-02-28,2025-03-31,2025-04-30,2025-05-31,2025-06-30,2025-07-31,2025-08-31,2025-09-30,2025-10-31,2025-11-30,2025-12-31,2026-01-31,2026-02-28,2026-03-31
ticker,,,,,,,,,,,,,,,,,
ABT,0.0,0.0,0.0,0.0,0.0000,0.0000,0.0,25.3733,0.0000,0.0,53.8098,0.000,0.0,60.6539,0.0000,0.0,39.7659
ADM,0.0,0.0,0.0,0.0,15.8174,0.0000,0.0,43.0667,0.0000,0.0,31.7230,0.000,0.0,52.9699,0.0000,0.0,0.0000
AMTX,0.0,0.0,0.0,0.0,14.1498,0.0000,0.0,53.1593,0.0000,0.0,56.4925,0.000,0.0,48.1025,0.0000,0.0,0.0000
ANDE,0.0,0.0,0.0,0.0,11.2328,0.0000,0.0,66.7899,0.0000,0.0,50.6272,0.000,0.0,57.4896,0.0000,0.0,0.0000
AVO,0.0,0.0,0.0,0.0,0.0000,10.6673,0.0,0.0000,67.3271,0.0,0.0000,56.859,0.0,0.0000,35.2564,0.0,0.0000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
VLO,0.0,0.0,0.0,0.0,0.0000,0.0000,0.0,14.8366,0.0000,0.0,67.1147,0.000,0.0,53.5976,0.0000,0.0,67.2061
WEST,0.0,0.0,0.0,0.0,9.3118,0.0000,0.0,54.7627,0.0000,0.0,51.1016,0.000,0.0,54.7663,0.0000,0.0,0.0000
WVVI,0.0,0.0,0.0,0.0,13.8711,0.0000,0.0,70.4418,0.0000,0.0,24.2292,0.000,0.0,73.6168,0.0000,0.0,0.0000
